# 🎯 TradeSight — Quick Intro Demo

TradeSight is a self-hosted Python paper trading strategy lab. This notebook shows the core mechanics:

- **15+ technical indicators** (RSI, MACD, Bollinger Bands, EMA crossovers, ATR)
- **Strategy backtesting** against historical OHLCV data
- **AI tournament system** — strategies compete, best ones survive

No API key required for this demo (uses yfinance for price data).

🔗 Full project: [github.com/rmbell09-lang/tradesight](https://github.com/rmbell09-lang/tradesight)

In [ ]:
# Install dependencies
!pip install yfinance pandas numpy matplotlib --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Fetch 6 months of SPY data
ticker = 'SPY'
df = yf.download(ticker, period='6mo', interval='1d', progress=False)
print(f"Downloaded {len(df)} days of {ticker} data")
df.tail()

In [ ]:
# --- Core indicators used by TradeSight ---

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = -delta.where(delta < 0, 0).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast).mean()
    ema_slow = series.ewm(span=slow).mean()
    macd = ema_fast - ema_slow
    signal_line = macd.ewm(span=signal).mean()
    return macd, signal_line

def compute_bollinger(series, period=20, std_dev=2):
    sma = series.rolling(period).mean()
    std = series.rolling(period).std()
    return sma + (std * std_dev), sma, sma - (std * std_dev)

close = df['Close'].squeeze()

df['RSI'] = compute_rsi(close)
df['MACD'], df['MACD_Signal'] = compute_macd(close)
df['BB_upper'], df['BB_mid'], df['BB_lower'] = compute_bollinger(close)
df['EMA_20'] = close.ewm(span=20).mean()
df['EMA_50'] = close.ewm(span=50).mean()

print("Indicators computed. Latest values:")
print(df[['RSI', 'MACD', 'MACD_Signal', 'EMA_20', 'EMA_50']].tail(3).round(2))

In [ ]:
# --- Simple backtest: EMA crossover strategy ---
# Buy when 20-EMA crosses above 50-EMA, sell when it crosses below

df = df.copy()
df['Signal'] = 0
df.loc[df['EMA_20'] > df['EMA_50'], 'Signal'] = 1   # long
df.loc[df['EMA_20'] < df['EMA_50'], 'Signal'] = -1  # flat

df['Returns'] = close.pct_change()
df['Strategy'] = df['Signal'].shift(1) * df['Returns']

# Cumulative performance
df['Cum_Buy_Hold'] = (1 + df['Returns']).cumprod()
df['Cum_Strategy'] = (1 + df['Strategy']).cumprod()

total_return_bh = (df['Cum_Buy_Hold'].iloc[-1] - 1) * 100
total_return_strat = (df['Cum_Strategy'].iloc[-1] - 1) * 100

print(f"Buy & Hold return:      {total_return_bh:+.2f}%")
print(f"EMA crossover return:   {total_return_strat:+.2f}%")

In [ ]:
# Visualize
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))

# Price + EMAs
ax1.plot(df.index, close, label='SPY Close', alpha=0.7)
ax1.plot(df.index, df['EMA_20'], label='EMA 20', linestyle='--')
ax1.plot(df.index, df['EMA_50'], label='EMA 50', linestyle='--')
ax1.set_title('SPY Price + EMA Crossover Strategy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# RSI
ax2.plot(df.index, df['RSI'], color='purple', label='RSI(14)')
ax2.axhline(70, color='red', linestyle='--', alpha=0.5, label='Overbought')
ax2.axhline(30, color='green', linestyle='--', alpha=0.5, label='Oversold')
ax2.set_ylim(0, 100)
ax2.set_title('RSI(14)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Cumulative returns
ax3.plot(df.index, df['Cum_Buy_Hold'], label='Buy & Hold', color='gray')
ax3.plot(df.index, df['Cum_Strategy'], label='EMA Crossover', color='blue')
ax3.set_title('Cumulative Returns')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n📌 In TradeSight, 9 strategies like this run simultaneously")
print("   overnight via AI tournaments — best performers survive.")
print("   Install: git clone https://github.com/rmbell09-lang/tradesight")

## ⬇️ Run TradeSight Locally

```bash
git clone https://github.com/rmbell09-lang/tradesight.git
cd tradesight
pip install -r requirements.txt
python START_TRADESIGHT.py
```

- Web dashboard: `http://localhost:5000`
- Runs paper trades via Alpaca (free account)
- AI tournaments run overnight via cron
- 100% local — your strategies stay private

⭐ [Star on GitHub](https://github.com/rmbell09-lang/tradesight) if this was useful!